### KNB Data Transformation: Great-tailed Grackle Behavioral Experiments (*Quiscalus mexicanus*)
---
**Capstone Project · DataTalks.Club 2026**  
**Author:** Victoria Torreno  
**Source:** [KNB Ecoinformatics](https://knb.ecoinformatics.org/view/doi:10.5063/F13B5XBC)  
**Goal:** Prototype Bronze → Silver transformations including type casting, schema standardization, column renaming, and deduplication before implementing in the production Spark job.

#### Setup Spark Session and Load Raw GCS Data

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import count, when, col
from dotenv import load_dotenv
from pathlib import Path

In [ ]:
load_dotenv()

In [ ]:
GCS_BUCKET = os.getenv('GCS_BUCKET')
JAR_PATH = str(Path().resolve().parents[1] / 'jars' / 'gcs-connector.jar')

In [ ]:
# initialize Spark session with GCS connector
spark = SparkSession.builder \
    .appName('knb_bronze_to_silver') \
    .config('spark.jars', JAR_PATH) \
    .config('spark.hadoop.fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem') \
    .config('spark.hadoop.fs.AbstractFileSystem.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS') \
    .config('spark.hadoop.google.cloud.auth.service.account.enable', 'true') \
    .config('spark.hadoop.google.cloud.auth.service.account.json.keyfile', os.getenv('GOOGLE_APPLICATION_CREDENTIALS')) \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark initialized with local GCS connector.")

In [ ]:
# read raw CSVs from GCS Bronze 
water_tube_df = spark.read \
    .option('header', 'false') \
    .option('inferSchema', 'true') \
    .csv(f'gs://{GCS_BUCKET}/bronze_knb/water_tube.csv')

In [ ]:
color_test_df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(f'gs://{GCS_BUCKET}/bronze_knb/color_test.csv')

In [ ]:
interaction_df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(f'gs://{GCS_BUCKET}/bronze_knb/interaction.csv')

In [ ]:
print(f'water_tube rows: {water_tube_df.count():,}')
print(f'color_test rows: {color_test_df.count():,}')
print(f'interaction rows: {interaction_df.count():,}')

#### Schema Enforcement & Column Naming
Assigning column names to `water_tube` (no header in source) and standardizing all three datasets to snake_case. Drops sparse and redundant columns identified during exploration.

In [ ]:
# water_tube has no header, assigning column names from KNB documentation
WATER_TUBE_COLUMNS = [
    'experiment', 'date', 'batch',
    'bird', 'trial', 'choice_number',
    'choice_correct', 'choice',
    'extracted_food', 'water_level',
    'tube_on_left', 'notes'
]

In [ ]:
water_tube_df = water_tube_df.toDF(*WATER_TUBE_COLUMNS)

In [ ]:
# standardize to snake_case
color_test_df = color_test_df.toDF(*[
    c.lower().replace(' ', '_').replace('?', '').strip()
    for c in color_test_df.columns
])
interaction_df = interaction_df.toDF(*[
    c.lower().replace(' ', '_').replace('?', '').strip()
    for c in interaction_df.columns
])

# verify columns
print('\ncolor_test columns:')
print('\n'.join(color_test_df.columns))

print('\ninteraction columns:')
print('\n'.join(interaction_df.columns))

In [ ]:
# drop sparse column identified during exploration
water_tube_df = water_tube_df.drop('notes')

In [ ]:
# drop columns with high null rate
# correctchoice is redundant, correct column is numeric equivalent
color_test_df = color_test_df.drop(
    'nonoverlappingwindow4-trialbins',
    'criterion',
    'preference_notes',
    'correctchoice'
)

In [ ]:
# rename color_test columns for clarity
color_test_df = color_test_df \
    .withColumnRenamed('coloronleft', 'color_on_left') \
    .withColumnRenamed('correct', 'is_correct')

In [ ]:
# rename interaction columns for clarity
interaction_df = interaction_df \
    .withColumnRenamed('optiononleft', 'option_on_left') \
    .withColumnRenamed('approachedfirst', 'approached_first') \
    .withColumnRenamed('putmorefoodon', 'put_more_food_on')

In [ ]:
print('water_tube columns:')
print('\n'.join(water_tube_df.columns))

print('\ncolor_test columns:')
print('\n'.join(color_test_df.columns))

print('\ninteraction columns:')
print('\n'.join(interaction_df.columns))

#### Preview Raw Data

In [ ]:
water_tube_df.limit(5).toPandas()

In [ ]:
color_test_df.limit(5).toPandas()

In [ ]:
interaction_df.limit(5).toPandas()

#### Type Casting
Casting date fields from string to date type, binary fields from string to boolean, and numeric fields to appropriate integer/double types across all three datasets.

In [ ]:
# water_tube
water_tube_df = water_tube_df \
    .withColumn('date', f.to_date(f.col('date'), 'd-MMM-yy')) \
    .withColumn('batch', f.col('batch').cast('integer')) \
    .withColumn('trial', f.col('trial').cast('integer')) \
    .withColumn('choice_number', f.col('choice_number').cast('integer')) \
    .withColumn('water_level', f.col('water_level').cast('double')) \
    .withColumn('choice_correct', (f.col('choice_correct') == 'Yes').cast('boolean')) \
    .withColumn('extracted_food', (f.col('extracted_food') == 'Yes').cast('boolean'))

water_tube_df.printSchema()

In [ ]:
# color_test
color_test_df = color_test_df \
    .withColumn('date', f.to_date(f.col('date'), 'd-MMM-yy')) \
    .withColumn('batch', f.col('batch').cast('integer')) \
    .withColumn('trial', f.col('trial').cast('integer')) \
    .withColumn('is_correct', f.col('is_correct').cast('boolean'))

color_test_df.printSchema()

In [ ]:
# interaction
interaction_df = interaction_df \
    .withColumn('date', f.to_date(f.col('date'), 'd-MMM-yy')) \
    .withColumn('trial', f.col('trial').cast('integer'))

interaction_df.printSchema()

#### Deduplication
Checking for and removing duplicate records across all three datasets.

In [ ]:
# create a list to track the processed DataFrames
processed_dfs = []

for name, df in [('water_tube', water_tube_df), 
                 ('color_test', color_test_df), 
                 ('interaction', interaction_df)]:
    
    initial_ct = df.count()
    df_clean = df.dropDuplicates()
    current_ct = df_clean.count()
    
    print(f'{name}: removed {initial_ct - current_ct:,} duplicates, {current_ct:,} remaining')
    processed_dfs.append(df_clean)

water_tube_df, color_test_df, interaction_df = processed_dfs

#### Null Audit
Checking null counts across all three datasets before writing to Silver.

In [ ]:
# water_tube null audit
null_counts = water_tube_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in water_tube_df.columns
])
null_counts.toPandas().T.rename(columns={0: 'null_count'})

In [ ]:
# color_test null audit
null_counts = color_test_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in color_test_df.columns
])
null_counts.toPandas().T.rename(columns={0: 'null_count'})

In [ ]:
# interaction null audit
null_counts = interaction_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in interaction_df.columns
])
null_counts.toPandas().T.rename(columns={0: 'null_count'})

#### Final Validation
A last look at the data before committing to GCS Silver.

In [ ]:
# subject trial counts per dataset
print('water_tube trials per bird:')
water_tube_df.groupBy('bird').count().orderBy('count', ascending=False).toPandas()

In [ ]:
print('color_test trials per bird:')
color_test_df.groupBy('bird').count().orderBy('count', ascending=False).toPandas()

In [ ]:
print('interaction trials per bird:')
interaction_df.groupBy('bird').count().orderBy('count', ascending=False).toPandas()

In [ ]:
# choice distribution (water_tube)
water_tube_df.groupBy('choice').count().orderBy('count', ascending=False).toPandas()

In [ ]:
# correct choice rate (color_test)
color_test_df.groupBy('is_correct').count().toPandas()

#### Write to GCS Silver
Writing all three cleaned and transformed datasets to GCS Silver as Parquet for downstream dbt modeling.

In [ ]:
SILVER_PATH = f'gs://{GCS_BUCKET}/silver_knb'

water_tube_df.write \
    .mode('overwrite') \
    .parquet(f'{SILVER_PATH}/water_tube')

color_test_df.write \
    .mode('overwrite') \
    .parquet(f'{SILVER_PATH}/color_test')

interaction_df.write \
    .mode('overwrite') \
    .parquet(f'{SILVER_PATH}/interaction')

print(f"Silver data written to: {SILVER_PATH}")
print(f"water_tube:  {water_tube_df.count():,} rows")
print(f"color_test:  {color_test_df.count():,} rows")
print(f"interaction: {interaction_df.count():,} rows")